In [3]:
import torch
from math import sqrt

OSError: [WinError 127] The specified procedure could not be found. Error loading "c:\Users\spres\anaconda3\envs\torch_gpu\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

Area of a Circle

In [6]:
torch.manual_seed(42)

r = torch.rand(size = (1000,1)) * 5

b = torch.pi * torch.square(r)
pow = 3
powers = torch.arange(start = 0, end = pow, step = 1)
B = torch.pow(r,powers)
p1 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p2 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p3 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p4 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p5 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p6 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p7 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p8 = torch.ones(size = (B.shape[1],1), requires_grad= True)

A = torch.concat((torch.ones_like(b), B @ p1, torch.sin(B @ p2), torch.cos(B @ p3), torch.special.erf(B @ p4), torch.special.airy_ai(B @ p5), torch.special.bessel_j0(B @ p6), torch.special.ndtr(B @ p7), torch.special.legendre_polynomial_p(B @ p8, 3)), dim = 1)
x = torch.ones(size = (A.shape[1], 1),requires_grad= True)

In [8]:
lamb = 3e-1
parameters = [x, p1, p2, p3, p4, p5, p6, p7, p8]

optimizer = torch.optim.Adam(params = parameters, lr = 3e-4)
B = torch.pow(r,powers)
A = torch.concat((torch.ones_like(b), B @ p1, torch.sin(B @ p2), torch.cos(B @ p3), torch.special.erf(B @ p4), torch.special.airy_ai(B @ p5), torch.special.bessel_j0(B @ p6), torch.special.ndtr(B @ p7), torch.special.legendre_polynomial_p(B @ p8, 3)), dim = 1)

COS_CIR_losses = []
COS_CIR_epochs = []
for i in range(100000):
    loss = torch.square(torch.linalg.vector_norm((A @ x - b), ord = 2)) + lamb * sum([j.abs().sum() for j in parameters])
    optimizer.zero_grad()
    loss.backward()
    COS_CIR_losses.append(loss.item())
    COS_CIR_epochs.append(i)
    optimizer.step()
    B = torch.pow(r,powers)
    A = torch.concat((torch.ones_like(b), B @ p1, torch.sin(B @ p2), torch.cos(B @ p3), torch.special.erf(B @ p4), torch.special.airy_ai(B @ p5), torch.special.bessel_j0(B @ p6), torch.special.ndtr(B @ p7), torch.special.legendre_polynomial_p(B @ p8, 3)), dim = 1)

print(loss.item())

COSSR_x = x.clone()

KeyboardInterrupt: 

In [4]:
def FISTA(Vandermonde, y, epochs = 1000, lamb = 0.5, grad = lambda A, x, b: (A.T @ (A @ x - b)), cutoff = 0.5, lr = None): ## lambda is arbitrarily set but you can change it. cutoff is the cutoff value for doing the least squares at the end.
  x = torch.rand(size = (Vandermonde.shape[1],)).reshape(-1,1)
  if lr == None:
    lr = 1 / (torch.linalg.norm(Vandermonde.T @ Vandermonde, ord = 2)) ## lr guareenteeing convergence for fista

  Vandermonde = Vandermonde
  soft_thres = lambda x, th = lamb * lr: torch.sign(x) * torch.maximum(torch.abs(x) - th, torch.tensor(0))
  prev_step = torch.zeros(Vandermonde.shape[1]).reshape(-1,1)

  tk = 1
  losses = []
  epochs_list = []
  for epoch in range(epochs):
    Beta = (tk - 1) / ((1 + sqrt(1 + 4 * tk ** 2)) / 2)
    prev_step = Beta * prev_step - lr * grad(Vandermonde, x + prev_step, y)
    x += prev_step
    x = soft_thres(x)

    tk = (1 + sqrt(1 + 4 * tk ** 2)) / 2
    loss = torch.linalg.vector_norm((A @ x - b), ord = 2).square() / (A @ x - b).size(0) + torch.linalg.norm(x, ord = 1)

    losses.append(loss.item())
    epochs_list.append(epoch)

  print(loss.item())
  return epochs_list, losses, x

  ## We are comparing without cutoff/adjustments
  active = np.abs(x) > cutoff

  x_debiased = np.zeros_like(x)
  x_debiased[active] = np.linalg.lstsq(
      Vandermonde[:, active.reshape(-1)],
      y,
      rcond=None
  )[0].reshape(-1)

  return x_debiased

In [5]:
A = torch.concat([B,torch.sin(r), torch.cos(r), torch.special.erf(r), torch.special.airy_ai(r), torch.special.bessel_j0(r), torch.special.ndtr(r), torch.special.legendre_polynomial_p(r,3)], dim = 1)
FISTA_CIR_epochs, FISTA_CIR_losses, FISTA_CIR_x = FISTA(A, b, epochs= 100000, lamb = 3e-1)

4.336974143981934


In [6]:
COSSR_Circle_p = [p.clone() for p in parameters]

Volume of a Cylinder

In [7]:
torch.manual_seed(42)

r = torch.rand(size = (100000,1)) * 5
h = torch.rand(size = (100000,1)) * 5
b = torch.pi * torch.square(r) * h

pow = 3
powers = torch.arange(start = 0, end = pow, step = 1)

B = (torch.pow(r,torch.meshgrid(powers, powers)[0].reshape(-1,)) * torch.pow(h,torch.meshgrid(powers, powers)[1].reshape(-1,)))
p1 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p2 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p3 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p4 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p5 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p6 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p7 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p8 = torch.ones(size = (B.shape[1],1), requires_grad= True)

A = torch.concat((torch.ones_like(b), B @ p1, torch.sin(B @ p2), torch.cos(B @ p3), torch.special.erf(B @ p4), torch.special.airy_ai(B @ p5), torch.special.bessel_j0(B @ p6), torch.special.ndtr(B @ p7), torch.special.legendre_polynomial_p(B @ p8, 3)), dim = 1)
x = torch.ones(size = (A.shape[1], 1), requires_grad= True)

parameters = [x, p1, p2, p3, p4, p5, p6, p7, p8]


c:\Users\spres\anaconda3\envs\torch_gpu\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
lamb = 3e-1
optimizer = torch.optim.Adam(params = parameters, lr = 3e-4)


B = (torch.pow(r,torch.meshgrid(powers, powers)[0].reshape(-1,)) * torch.pow(h,torch.meshgrid(powers, powers)[1].reshape(-1,)))
A = torch.concat((torch.ones_like(b), B @ p1, torch.sin(B @ p2), torch.cos(B @ p3), torch.special.erf(B @ p4), torch.special.airy_ai(B @ p5), torch.special.bessel_j0(B @ p6), torch.special.ndtr(B @ p7), torch.special.legendre_polynomial_p(B @ p8, 3)), dim = 1)

COS_CYL_losses = []
COS_CYL_epochs = []

for i in range(100000):
    loss = torch.linalg.vector_norm((A @ x - b), ord = 2).square() / (A @ x - b).size(0) + lamb * sum([j.abs().sum() for j in parameters])

    COS_CYL_losses.append(loss.item())
    COS_CYL_epochs.append(i)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(loss.item())

    B = (torch.pow(r,torch.meshgrid(powers, powers)[0].reshape(-1,)) * torch.pow(h,torch.meshgrid(powers, powers)[1].reshape(-1,)))
    A = torch.concat((torch.ones_like(b), B @ p1, torch.sin(B @ p2), torch.cos(B @ p3), torch.special.erf(B @ p4), torch.special.airy_ai(B @ p5), torch.special.bessel_j0(B @ p6), torch.special.ndtr(B @ p7), torch.special.legendre_polynomial_p(B @ p8, 3)), dim = 1)

print(loss.item())

7.221867561340332


In [9]:
COSSR_CYL_p = [p.clone() for p in parameters]

In [10]:
A = torch.concat([B,torch.sin(r), torch.cos(r), torch.special.erf(r), torch.special.airy_ai(r), torch.special.bessel_j0(r), torch.special.ndtr(r), torch.special.legendre_polynomial_p(r,3)], dim = 1)
FISTA_CYL_epochs, FISTA_CYL_losses, FISTA_CYL_x = FISTA(A, b, epochs= 100000, lamb = 3e-1)

4.420440673828125


Rediscovering $sin(c^2 + d^2)$

In [11]:
torch.manual_seed(42)

c = (5) * torch.rand(size = (100000,1)) - 2.5
d = (5) * torch.rand(size = (100000,1)) - 2.5

b = torch.sin(c ** 2 + d ** 2)

pow = 3
powers = torch.arange(start = 0, end = pow, step = 1)

B = (torch.pow(c,torch.meshgrid(powers, powers)[0].reshape(-1,)) * torch.pow(d,torch.meshgrid(powers, powers)[1].reshape(-1,)))
p1 = torch.ones(size = (B.shape[1],1), requires_grad= True)
p2 = torch.ones(size = (B.shape[1],1), requires_grad= True)


A = torch.concat((torch.ones_like(b), B @ p1, torch.sin(B @ p2)), dim = 1)
x = torch.ones(size = (A.shape[1], 1), requires_grad= True)

In [ ]:
total_parameters = [x, p1, p2]

idx = 0
parameters = [total_parameters[idx]]

lamb = 3e-4

COS_SIN_losses = []
COS_SIN_epochs = []

for i in range(100000):
    if i % 20 == 0:
        idx = (idx + 1) % len(total_parameters)
        parameters = [total_parameters[idx]]
        optimizer = torch.optim.Adam(params = parameters, lr = 3e-3)

    B = (torch.pow(c,torch.meshgrid(powers, powers)[0].reshape(-1,)) * torch.pow(d,torch.meshgrid(powers, powers)[1].reshape(-1,)))
    A = torch.concat((torch.ones_like(b), B @ p1, torch.sin(B @ p2)), dim = 1)
    loss = torch.linalg.vector_norm((A @ x - b), ord = 2).square() / (A @ x - b).size(0) + lamb * sum([j.abs().sum() for j in total_parameters])

    COS_SIN_losses.append(loss.item())
    COS_SIN_epochs.append(i)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(loss.item())

In [ ]:
COSSR_SIN_p = [p.clone() for p in total_parameters]

In [ ]:
## equivalent formula for SSR
A = torch.concat([B,torch.sin(c), torch.sin(d), torch.sin(c * d), torch.sin(c ** 2), torch.sin(d ** 2), torch.sin(c**2 * d), torch.sin(d**2 * c), torch.sin(c**2 * d**2), torch.sin(c + d), torch.sin(c**2 + d), torch.sin(c + d**2), torch.sin(c**2 + d **2), torch.sin(c * d + d), torch.sin(c**2 * d + d), torch.sin(c + d**2 * c), torch.sin(c**2 * d + d**2 * c), torch.sin(2 * c**2 * d**2)], dim = 1)
FISTA_SIN_epochs, FISTA_SIN_losses, FISTA_SIN_x = FISTA(A, b, epochs= 100000, lamb = 3e-1)

0.9999999403953552


In [20]:
fig, axes = plt.subplots(3,2, figsize = (12,6))

axes[0][0].plot(COS_CIR_epochs,COS_CIR_losses)
axes[0][1].plot(FISTA_CIR_epochs,FISTA_CIR_losses)
axes[1][0].plot(COS_CYL_epochs,COS_CYL_losses)
axes[1][1].plot(FISTA_CYL_epochs,FISTA_CYL_losses)
axes[2][0].plot(COS_SIN_epochs,COS_SIN_losses)
axes[2][1].plot(FISTA_SIN_epochs, FISTA_SIN_losses)

: 